In [16]:
import requests
from bs4 import BeautifulSoup
import re

def findTotalRecords(soup):
    # Find the <script> tag containing the totalRecords
    script_tag = soup.find('script', string=re.compile('totalRecords'))

    total_records = 0
    if script_tag:
        print(script_tag)
        match = re.search(r'totalRecords\s*:\s*(\d+)', script_tag.string)
        # totalRecords: 52,
        if match: total_records = int(match.group(1))
    
    return total_records

def fetch(url):

    session = requests.Session() # use same session to allow page switching

    response = session.get(url)
    content = response.content
    
    total_records = findTotalRecords(BeautifulSoup(content, "lxml"))
    if total_records == 300: print("NOTE: MAX RECORD COUNT HIT WITH URL", url)

    # 0-50 records means no additional pages, 51-100 means 1 additional page, etc.
    additional_pages = max(0, total_records // 50 - (not (total_records % 50))) # note 0 gives -1
    pageUrl = "https://more.app.vanderbilt.edu/more/SearchClassesExecute!switchPage.action?pageNum="

    for i in range(additional_pages):
        add_response = session.get(pageUrl + str(i + 1))
        content += add_response.content
    
    soup = BeautifulSoup(content, "lxml")
    return soup

In [ ]:
# usage

url = "https://more.app.vanderbilt.edu/more/SearchClassesExecute!search.action?keywords=" + "101"

sample_soup = fetch(url)

print(sample_soup.prettify()) # sanitized html
# print(sample_soup.get_text()) # text from html

<html>
 <body>
  <div id="currentPage">
   <div id="paginator">
   </div>
   <div id="paginatorReport">
   </div>
   <div class="clear">
   </div>
  </div>
  <script>
   YAHOO.util.Event.onDOMReady( function () {
		   Nifty("div.classHeaderRounded","small");
		   // Add hovers to paginators
		   YAHOO.mis.student.Util.doForEachElement(".yui-pg-last",
		   function(element) {
		      element.setAttribute( "title" , "Jump to the last page of results" ); 
		   });		   
		   YAHOO.mis.student.Util.doForEachElement(".yui-pg-first",
		   function(element) {
			  element.setAttribute( "title" , "Jump to the first page of results" ); 
		   });
		   YAHOO.mis.student.Util.doForEachElement(".yui-pg-next",
		   function(element) {
			  element.setAttribute( "title" , "Jump to the next page of results" ); 
		    });
		   YAHOO.mis.student.Util.doForEachElement(".yui-pg-previous",
		   function(element) {
			  element.setAttribute( "title" , "Jump to the previous page of results" ); 
		   });
		});

In [22]:
# for simple testing

base_url = "https://more.app.vanderbilt.edu/more/SearchClassesExecute!search.action?keywords="

for i in range(101, 102, 1):
    addon = f"{i:03d}"
    url = base_url + addon

    # session = requests.Session() # use same session to allow page switching
    # response = session.get(url + addon)
    response = requests.get(url)
    content = response.content

    total_records = findTotalRecords(BeautifulSoup(content, "lxml"))

    soup = fetch(url)

<script>
		YAHOO.util.Event.onDOMReady( function () {
		   Nifty("div.classHeaderRounded","small");
		   // Add hovers to paginators
		   YAHOO.mis.student.Util.doForEachElement(".yui-pg-last",
		   function(element) {
		      element.setAttribute( "title" , "Jump to the last page of results" ); 
		   });		   
		   YAHOO.mis.student.Util.doForEachElement(".yui-pg-first",
		   function(element) {
			  element.setAttribute( "title" , "Jump to the first page of results" ); 
		   });
		   YAHOO.mis.student.Util.doForEachElement(".yui-pg-next",
		   function(element) {
			  element.setAttribute( "title" , "Jump to the next page of results" ); 
		    });
		   YAHOO.mis.student.Util.doForEachElement(".yui-pg-previous",
		   function(element) {
			  element.setAttribute( "title" , "Jump to the previous page of results" ); 
		   });
		});
		var pag = new YAHOO.widget.Paginator({
		   initialPage : 1,
		   rowsPerPage : 50,
		   totalRecords: 52,
		   containers  : [ 'paginator', 'bottomPaginato

In [20]:
# parsing data (based off code block above)

import json
from datetime import datetime

data = []

# course codes (ex. CS 1101)
course_code_elements = soup.find_all('span', class_='classAbbreviation')

course_code_elements = course_code_elements[18]
for course_code in course_code_elements:

    # ex: Computer Science
    subject_name = course_code.find_previous('div', class_='subjectHeader')
    # name, number = course_code.get_text(strip=True).rstrip(':').split(' ')
    # number = int(number) # edge cases: 1600L, 2500D

    # ex: Programming and Problem Solving
    course_name = course_code.find_next('span', class_='classDescription')

    class_section = course_name.find_next('td', class_='classSection')

    credit_hours = class_section.find_next('td', class_='classHours')

    class_type = credit_hours.find_next('td', class_='classType')

    availability = class_type.find_next('a')
    if "availableStatus" in availability.get('class'):
        status = 0 # note that we NEVER get here if waitlist is nonempty
    elif "waitlistStatus" in availability.get('class'):
        status = 1
    elif "closedStatus" in availability.get('class'):
        status = 2
    else:
        # availability.get() returns a list
        raise ValueError(f"Class availability status unaccounted for: {availability.get('class')}")
    
    occupied, capacity = map(int, availability.get_text(strip=True).split('/'))

    # can be TBA
    meeting_days = availability.find_next('td', class_='classMeetingDays')

    # TODO: edge case: TBA, multiple listings
    meeting_times = meeting_days.find_next('td', class_='classMeetingTimes')

    # # Find all time slots (09:00a - 12:00p) and corresponding dates (02/02/2025 - 02/02/2025)
    # times = meeting_times.find_all(string=lambda text: text and 'a - p' in text)
    # dates = meeting_times.find_all('div', class_='randomDates')

    # # Extract the time and date pairs
    # time_date_pairs = []
    # for time, date in zip(times, dates):
    #     time_date_pairs.append((time.strip(), date.get_text().strip()))

    # # Print the pairs
    # for pair in time_date_pairs:
    #     print(pair)
    print(meeting_times)
    """
    # convert meeting time to 24-hr numerical values
    meeting_times_str = meeting_times.get_text(strip=True).split('\n')[0]
    start_time_str, end_time_str = meeting_times_str.strip().upper().split(' - ')
    start_time = datetime.strptime(start_time_str + "M", '%I:%M%p').strftime('%H:%M')
    end_time = datetime.strptime(end_time_str + "M", '%I:%M%p').strftime('%H:%M')
    
    date_range = meeting_times.find('div', class_='randomDates')
    
    # NOTE: this is usually dynamically loaded in so will likely be None
    location = meeting_times.find_next('td', class_='classBuilding')
    """
    location = meeting_days.find_next('td', class_='classBuilding')

    instructors = location.find_next('td', class_='classInstructor')

    # checking for note
    class_action_dummy = instructors.find_next('td', class_='classActions')
    next_sibling = class_action_dummy.find_next('tr')
    note = None
    if next_sibling and next_sibling.get('id', '').endswith('_notes'):
        note = next_sibling.find('td', colspan='9').get_text().strip()


    current_data = {
        "subject_name" : subject_name.find('h4').get_text(strip=True),
        # "course_code_name" : name,
        # "course_code_number" : number,
        "course_code" : course_code.get_text(strip=True).rstrip(":"),
        "course_name" : course_name.get_text(strip=True),
        "class_section" : class_section.get_text(strip=True),
        "credit_hours" : float(re.sub(r'[^\d.]+', '', credit_hours.get_text(strip=True))),
        "class_type" : class_type.get_text(strip=True),
        "availability": {
            "status": status,
            "occupied": occupied,
            "capacity": capacity
        },
        "meeting_days" : meeting_days.get_text(strip=True).replace('<br/>', ''),
        "meeting_start" : start_time,
        "meeting_end" : end_time,
        "date_range" : date_range.get_text(strip=True) if date_range else None,
        "location" : location.get_text(strip=True) if location.get_text(strip=True) else None,
        "instructors" : [instructor.strip() for instructor in instructors.get_text(strip=True).split('|')],
        "note" : note
    }

    data.append(current_data)

filename = "data.json"
with open(filename, "w") as file:
    json.dump(data, file, indent=4)

<td class="classMeetingTimes" onclick="YAHOO.mis.student.Topics.showClassDetailPanel.fire({classNumber : '10582', termCode : '1040', hideAddToCartButton : 'false'})">
         
            
               07:15p - 09:15p<br/>
<div class="randomDates">01/30/2025 - 01/30/2025</div>
               
            
               04:30p - 07:30p<br/>
<div class="randomDates">01/31/2025 - 01/31/2025</div>
               
            
               09:00a - 12:00p<br/>
<div class="randomDates">02/01/2025 - 02/01/2025</div>
               
            
               01:00p - 04:00p<br/>
<div class="randomDates">02/01/2025 - 02/01/2025</div>
               
            
               09:00a - 12:00p<br/>
<div class="randomDates">02/02/2025 - 02/02/2025</div>
</td>
